## Team
Aden Thompson

Cody Reep

10/02/2026

## Description

In this project, our team loaded a dataset containing ~2m data entries, 9 features each, on individual household electrical power consumption from the UC Irvine Machine Learning repository. Some features included date, time, voltage, and global active power.

All features were continuous except for date. We created a temporary date-time feature from the time and date features. The new date-time feature was then split into individual numeric year, month, and day features. We then dropped the original time and date features and temporary date-time feature.

We used sklearn’s linear regression model to predict global active power based on the other features. Cross-validation (10) showed a mean root mean square error (RMSE) of 0.04036 kW on a range from 0.08 (min) to 11.12 kW (max). Similarly, on the testing set, the model achieved an RMSE of 0.04035.





## Dataset Link

https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption

## Import Libraries

In [ ]:
import pandas as pd
from pandas.plotting import scatter_matrix
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score

## Get Data

In [ ]:
# https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption
data_url = "https://archive.ics.uci.edu/static/public/235/data.csv"
def load_power_data():
    csv_path = data_url
    return pd.read_csv(csv_path, sep=',', low_memory=False)

In [ ]:
# Set the variable power equal to the dataframe returned by load_power_data()
power = load_power_data()

## Data Inspection

### Check the data type of the columns

In [ ]:
# View the dataframes info
power.info()

### Convert the data type of the columns

In [ ]:
# Convert the data type of the columns to numeric, coercing errors to NaN
power["Sub_metering_1"] = pd.to_numeric(power["Sub_metering_1"], errors='coerce')
power["Sub_metering_2"] = pd.to_numeric(power["Sub_metering_2"], errors='coerce')
power["Sub_metering_3"] = pd.to_numeric(power["Sub_metering_3"], errors='coerce')
power["Voltage"] = pd.to_numeric(power["Voltage"], errors='coerce')
power["Global_active_power"] = pd.to_numeric(power["Global_active_power"], errors='coerce')
power["Global_reactive_power"] = pd.to_numeric(power["Global_reactive_power"], errors='coerce')
power["Global_intensity"] = pd.to_numeric(power["Global_intensity"], errors='coerce')

# Combine the Date and Time columns into a single datetime column > feature engineering
power['DateTime'] = pd.to_datetime(power['Date'] + ' ' + power['Time'], errors='coerce', format="%d/%m/%Y %H:%M:%S")


In [ ]:
# Get the year, month, and day from the DateTime column
power["year"] = power["DateTime"].dt.year
power["month"] = power["DateTime"].dt.month
power["day"] = power["DateTime"].dt.day

In [ ]:
# View the dataframes info
power.info()

## Summary Inspection

In [ ]:
# Get the shape of the dataframe
power.shape

In [ ]:
# View the beginning of the dataframe
power.head()

In [ ]:
# View the end of the dataframe
power.tail()

In [ ]:
# Get a description of the data rounded to the nearest 2 decimal places
power.describe(include='all').round(2)

In [ ]:
# Get a description of the data rounded to the nearest 2 decimal places
power.describe().round(2)

### Visualize the data

In [ ]:
# Visualize the distributions using histograms
power.hist(bins=50, figsize=(20,15))

In [ ]:
# Visualize the correlation between the features
attributes = ["Global_active_power", "Global_reactive_power", "Voltage", "Global_intensity", "Sub_metering_1", "Sub_metering_2", "Sub_metering_3"]
scatter_matrix(power[attributes], figsize=(12, 8))

## Preprocessing

### Data Cleaning

In [ ]:
# Fill in missing values with the mean using an imputer
imputer = SimpleImputer(strategy="median")
imputer.fit(power[attributes])
power[attributes] = imputer.transform(power[attributes])

In [ ]:
# Check for missing values
power.isna().sum()

In [ ]:
# Visualize the correlation between the features
scatter_matrix(power[["Global_active_power", "month", "year", "day", "Voltage"]], figsize=(12, 8))

In [ ]:
# View the dataframes info
power.info()

In [ ]:
# Drop the Date, Time, and DateTime columns from the entire data set
power = power.drop(["Date", "Time", "DateTime"], axis=1)

## Split the data into train and test sets

In [ ]:
# Split the dataset into train and test sets 80/20
# Could use stratified split

train_set, test_set = train_test_split(power, test_size=0.2, random_state=24)

## Apply Linear Regression Model

In [ ]:
# Get the features (columns used in prediction) and the target (what you are trying to predict) from the test set
power_features = train_set.drop("Global_active_power", axis=1) # Drop Global_active_power column for features
power_labels = train_set["Global_active_power"].copy() # Copy Global_active_power column for labels

In [ ]:
#Use a Linear Regression model
lin_reg = LinearRegression()
# Training the model on the features and the target
lin_reg.fit(power_features, power_labels)

## Accuracy of the model

In [ ]:
#Get the RMSE for the model
gPower_predictions = lin_reg.predict(power_features)
lin_mse = mean_squared_error(power_labels, gPower_predictions)
lin_rmse = np.sqrt(lin_mse)
lin_rmse

In [ ]:
#Get the MAE for the model
lin_mae = mean_absolute_error(power_labels, gPower_predictions)
lin_mae

## Better Evaluation Using Cross-Validation

In [ ]:
# Cross-validation with 10 folds
scores = cross_val_score(lin_reg, power_features, power_labels,
                         scoring="neg_mean_squared_error", cv=10)
# Get the RMSE for the model
lin_reg_rmse_scores = np.sqrt(-scores)

In [ ]:
# Display the scores, score mean, and score standard deviation
def display_scores(scores):
    print("Scores:", scores)
    print("Mean:", scores.mean())
    print("Standard deviation:", scores.std())

display_scores(lin_reg_rmse_scores)

## Apply the model to the test set

In [ ]:
# Get the features (columns used in prediction) and the target (what you are trying to predict) from the test set
X_test = test_set.drop("Global_active_power", axis=1) # Drop Global_active_power column for features
Y_test = test_set["Global_active_power"].copy() # Copy Global_active_power column for labels

In [ ]:
# Run the model on the test set
final_predictions = lin_reg.predict(X_test)

In [ ]:
# Get the RMSE for the model on the test set
final_mse = mean_squared_error(Y_test, final_predictions)
final_rmse = np.sqrt(final_mse)
final_rmse

In [ ]:
# Get the MAE for the model on the test set
final_mae = mean_absolute_error(Y_test, final_predictions)
final_mae